In [ ]:
import os
import sys
import pandas as pd

# 将项目根目录加入 Python 路径
current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir, '..'))
if os.path.exists(os.path.join(project_root, 'config.py')):
    if project_root not in sys.path:
        sys.path.insert(0, project_root)
        print(f"✅ 已添加项目根目录: {project_root}")

from config import get_driver
from src.retriever import find_matching_phages, find_similar_cases

print("✅ 模块导入完成")

In [ ]:
def analyze_cross_case_reuse(driver, case_a_id, case_b_id):
    """
    分析病例 B 是否复用了病例 A 的噬菌体经验。
    返回详细的验证报告。
    """
    # 1. 获取病例 A 和病例 B 的详细信息
    with driver.session() as session:
        result = session.run("""
            MATCH (c:ClinicalCase {case_id: $case_id})-[:INVOLVES_PATHOGEN]->(p:Pathogen)
            RETURN c.case_id AS case_id, 
                   p.species AS species, 
                   c.infection_type AS infection_type,
                   c.phage_treatment AS phage_treatment,
                   c.clinical_outcome AS outcome
        """, case_id=case_a_id)
        case_a = result.single()
        
        result = session.run("""
            MATCH (c:ClinicalCase {case_id: $case_id})-[:INVOLVES_PATHOGEN]->(p:Pathogen)
            RETURN c.case_id AS case_id, 
                   p.species AS species, 
                   c.infection_type AS infection_type,
                   c.phage_treatment AS phage_treatment,
                   c.clinical_outcome AS outcome
        """, case_id=case_b_id)
        case_b = result.single()
    
    if not case_a or not case_b:
        return {"error": "病例不存在，请检查 ID"}
    
    # 2. 解析病例 A 使用过的噬菌体（从 phage_treatment 属性提取）
    phages_used_in_a = []
    if case_a['phage_treatment']:
        phages_used_in_a = [x.strip() for x in case_a['phage_treatment'].split(',') if x.strip()]
    
    # 3. 获取病例 B 的相似病例（可能包含病例 A）
    similar_cases_for_b = find_similar_cases(
        driver, 
        case_b['species'], 
        case_b['infection_type'], 
        limit=10
    )
    
    # 4. 检查病例 A 是否在病例 B 的相似病例列表中
    case_a_in_similar = any(c['case_id'] == case_a_id for c in similar_cases_for_b)
    
    # 5. 获取病例 B 的匹配噬菌体列表（来自 Knowledge Layer）
    matching_phages = find_matching_phages(
        driver,
        case_b['species'],
        case_b.get('resistance', 'MDR'),  # 如果有 resistance 字段更好
        limit=50
    )
    
    # 6. 检查病例 A 使用的噬菌体是否出现在病例 B 的匹配列表中
    reused_phages = []
    for phage_name in phages_used_in_a:
        for mp in matching_phages:
            if mp['name'] == phage_name:
                reused_phages.append({
                    'name': phage_name,
                    'evidence_level': mp['evidence_level'],
                    'probability': mp['infection_probability']
                })
    
    # 7. 生成报告
    report = {
        "case_a": {
            "id": case_a_id,
            "species": case_a['species'],
            "infection_type": case_a['infection_type'],
            "phages_used": phages_used_in_a,
            "outcome": case_a['outcome']
        },
        "case_b": {
            "id": case_b_id,
            "species": case_b['species'],
            "infection_type": case_b['infection_type'],
            "outcome": case_b['outcome']
        },
        "is_case_a_similar_to_b": case_a_in_similar,
        "reused_phages": reused_phages,
        "reuse_count": len(reused_phages),
        "is_reuse_valid": len(reused_phages) > 0
    }
    
    return report

In [ ]:
# 获取数据库连接
driver = get_driver()

# 选择两个病例进行检测
CASE_A = "CASE-001"  # 源病例：李桂东（E. coli UTI，使用 CP-p-BC-23086/23062）
CASE_B = "CASE-003"  # 目标病例：另一个 E. coli UTI 病例

print(f"🔍 正在检测病例 {CASE_B} 是否复用了病例 {CASE_A} 的经验...\n")

report = analyze_cross_case_reuse(driver, CASE_A, CASE_B)

In [ ]:
if "error" in report:
    print(f"❌ {report['error']}")
else:
    print("=" * 60)
    print("📋 跨病例复用验证报告")
    print("=" * 60)
    
    print("\n📌 病例 A（源病例）：")
    print(f"   ID: {report['case_a']['id']}")
    print(f"   病原菌: {report['case_a']['species']}")
    print(f"   感染类型: {report['case_a']['infection_type']}")
    print(f"   临床结局: {report['case_a']['outcome']}")
    print(f"   使用的噬菌体: {report['case_a']['phages_used']}")
    
    print("\n📌 病例 B（目标病例）：")
    print(f"   ID: {report['case_b']['id']}")
    print(f"   病原菌: {report['case_b']['species']}")
    print(f"   感染类型: {report['case_b']['infection_type']}")
    print(f"   临床结局: {report['case_b']['outcome']}")
    
    print("\n🔗 相似性判断：")
    if report['is_case_a_similar_to_b']:
        print(f"   ✅ 病例 {CASE_A} 出现在病例 {CASE_B} 的相似病例列表中")
    else:
        print(f"   ⚠️ 病例 {CASE_A} 未出现在病例 {CASE_B} 的相似病例列表中（可能感染类型或病原菌不同）")
    
    print("\n🧬 知识复用检测：")
    if report['reuse_count'] > 0:
        print(f"   ✅ 检测到 {report['reuse_count']} 个噬菌体被复用：")
        for p in report['reused_phages']:
            print(f"      - {p['name']} (L{p['evidence_level']}, 概率: {p['probability']})")
        print(f"\n   🎯 结论：病例 {CASE_A} 的噬菌体治疗经验对病例 {CASE_B} 具有参考价值（V2 验证通过）")
    else:
        print(f"   ❌ 未检测到噬菌体复用")
        print(f"\n   💡 提示：尝试选择更相似的病例对（如同属 E. coli UTI 或 CRAB VAP）")
    
    print("\n" + "=" * 60)

In [ ]:
# 如果能检测到复用，尝试构建完整的 Learning Flywheel 链路
if report.get('reuse_count', 0) > 0:
    print("\n🔄 Learning Flywheel 闭环链路（V3 演示）：")
    print(f"   1️⃣ 病例 {CASE_A} 使用噬菌体进行治疗，结局: {report['case_a']['outcome']}")
    print(f"   2️⃣ 该治疗经验被编码为 Knowledge Layer 中的 L3 证据（单例临床验证）")
    print(f"   3️⃣ 病例 {CASE_B} 检索时，成功匹配到 {report['reuse_count']} 个噬菌体")
    print(f"   4️⃣ 这证明：上一个病例的经验可以被编码、被检索、被复用 ✅")
    print("\n   🎯 V3 验证通过：存在 1 条完整的可演示 Learning 闭环链路")
else:
    print("\n💡 尝试选择更合适的病例对进行 V3 验证：")
    print("   建议选择同一种病原菌（如 E. coli 或 A. baumannii）且感染类型相同（如 UTI 或 VAP）的病例")

In [ ]:
driver.close()
print("🔒 数据库连接已关闭")